## Shuffling — the cost and how to cut it

**Symptom in the UI:** stage duration dominated by *Shuffle Write Time* / *Shuffle Read Time*; total shuffle write running into hundreds of GB. The network is the bottleneck.

**Diagnosis:** joins and aggregations need all rows for a key on the same executor, which forces a **shuffle** — a redistribution of data across the network. A shuffle is the single most expensive thing most jobs do.

**Remedies, in priority order:**

- **Broadcast** when one side fits in memory — no shuffle at all (module 04).
- **Filter and project early** — the smallest shuffle is the one that moves the fewest bytes. Push `WHERE` and `SELECT` upstream so the shuffle carries only what's needed.
- **Right-size `spark.sql.shuffle.partitions`** — for a 1 TB shuffle, 200 partitions = 5 GB each (too big, spills); ~4000 = 250 MB each (the sweet spot).
- **Let AQE coalesce** — `spark.sql.adaptive.coalescePartitions.enabled = true` merges tiny post-shuffle partitions into right-sized ones automatically.
- **Avoid needless `repartition(N)`** before a join the optimiser would have shuffled correctly anyway — that's a double shuffle.

**The mental model:** you can't always avoid a shuffle, but you can shrink it (move fewer bytes) and balance it (right-size partitions). Broadcast avoids it; filter-early shrinks it; partition-sizing and AQE balance it.
